# Equity Risk Premia — v0.13: dynamic prices, shrinkage & cross-asset risks

This notebook retains the v0.12.2 experiments and adds dynamic mean-pricing forecasts, Gaussian shrinkage, a constrained balanced-panel RP-PCA benchmark, three-pass observed-factor prices with synchronized block inference, and actual funded Rates / traded ETF cross-asset return proxies. Historical evaluation is retrospective, not independent confirmation. All prices/returns are monthly decimals unless explicitly annualized. Revised source vintages, not point-in-time data.

# Equity Risk Premia — v0.12.2: rolling RP economic mapping

Preserves the original v0.12.1 research and adds a dedicated 2006–2011 formation / 2012 onward validation for **all six rolling factors** and FF/AQR. JK contemporaneous, lagged and pricing tests share an identical complete-case calendar. FRED files are validated for all six series before use. FRED changes remain proxies, not identified surprises. The new split is retrospective, not independent confirmation of RP12_PC2.

**v0.12.1 JK data-safety hotfix:** The original author monthly CSV has months with `MP_median + CBI_median != pc1_hf`, often when high-frequency stock data are missing. The loader logs these dates and treats their shocks as missing rather than assuming zero. The factor/PCA/pricing methodology is unchanged.

# Equity Risk Premia — v0.12: Stability, priced factors and identified shocks

This standalone notebook **preserves the v0.11 published FF6/BAB/QMJ & residual-factor experiments** and adds three strictly distinct research blocks:

1. **Rolling residual & pricing-weighted PCs**, with each month traded using only previously observable portfolio returns.
2. **Common-sample Fama–MacBeth and two-step shrinkage GMM/SDF**, comparing holdout pricing using fixed formation-only portfolios, HAC and paired time-block bootstrap.
3. **Public high-frequency identified policy surprises:** Jarociński–Karadi monthly MP & CBI shock decomposition. Separate option to load genuine CPI-release surprises constructed from actual minus pre-announcement consensus.

**Important:** The RP method is a *residual/FF-conditional augmented-mean adaptation* of RP-PCA, **not** the complete Lettau–Pelger estimators and inference. FMB does not include Shanken corrections and GMM with shrinkage does not support standard J-test inference. Characteristic-sorted holdout portfolios can share constituent stocks. The high-frequency Fed shock identification relies on event-window/sign restrictions; no series of CPI surprises is inferred from FRED-level CPI. All results are research diagnostics, not deployed trading premia.

In [1]:
import sys,base64,gzip,tarfile,io
from pathlib import Path
EMBEDDED="..."
RUNTIME=Path('/content/erp_v013_notebook_runtime') if Path('/content').exists() else Path.cwd()/'erp_v013_notebook_runtime'
RUNTIME.mkdir(parents=True,exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(EMBEDDED))),mode='r:') as tf:
    tf.extractall(RUNTIME/'src',filter='data' if sys.version_info>=(3,12) else None)
sys.path.insert(0,str(RUNTIME/'src'))
# Avoid reusing a previously imported ERP in the same Colab runtime.
for key in list(sys.modules):
    if key=='erp' or key.startswith('erp.'): del sys.modules[key]
import numpy as np,pandas as pd
from erp import (__version__,EquityData,analyze_equity_residuals,
    analyze_equity_factor_extensions,load_equity_fred_macro,
    analyze_equity_v012,load_jk_monetary_shocks,
    load_inflation_release_surprises,analyze_equity_shock_mapping,
    estimate_jk_shock_prices)
print('Embedded ERP source:', __version__)

from erp import analyze_rolling_shocks
from erp.equity_macro import find_complete_equity_fred_file
assert __version__ == "0.13.0"

from erp import (analyze_dynamic_premia,constrained_rp_rolling,gx_initial_inference,
    economic_return_decomposition,load_etf_adjusted_prices,etf_factor_returns,
    load_rates_factor_returns,managed_factor_returns,RatesData,RatesPremiaEngine)
from erp.equity_extensions import _aligned
from erp.equity_residual import _hac_alpha,_holm


Embedded ERP source: 0.13.0


## 1. Fixed discovery / test design

First 80 portfolios: 25 Size–B/M, 30 Industries, 25 Size–Momentum. Holdout 50: 25 Size–Profitability + 25 Size–Investment. **Constituent stocks can overlap** despite distinct portfolio definitions. Train through December 2005, OOS from January 2006. `GAMMAS` and `N_PCS` **must not be selected by validation performance**.


In [2]:
MODEL='FF6_BAB_QMJ'
AS_OF='2026-10-31'
TRAIN_END='2005-12-31'
DISCOVERY=('SIZE_BM_25','INDUSTRY_30','SIZE_MOM_25')
VALIDATION=('SIZE_OP_25','SIZE_INV_25')
N_PCS=5
GAMMAS=(0.,12.,60.) # predeclared gamma=0 is the original PCA baseline
NULL_DRAWS=199
NULL_SEED=7331
MACRO_STATE_LAG_MONTHS=2
AQR_PATHS={} # original AQR workbook fallback if source blocked
FRED_LOCAL_PATH=None # OPTIONAL: original combined FRED CSV/ZIP from browser
CACHE=Path('erp_cache')
OUT=Path('erp_equity_v013_results');OUT.mkdir(parents=True,exist_ok=True)

SHOCK_TRAIN_END="2011-12-31"
PROMPT_FRED_UPLOAD=True # set False for noninteractive runs

# New experiments: frozen hyperparameters, no OOS tuning.
DYNAMIC_WINDOW=72
DYNAMIC_MIN_TRAIN=60
DYNAMIC_REFIT=3
DYNAMIC_GX_K=8
DYNAMIC_BOOT_DRAWS=199
RUN_CROSS_ASSET_ETFS=True
RUN_FUNDED_RATES=True
ETF_ADJUSTED_CSV=None # optional local CSV with Date,IEF,HYG,UUP,DBC,VIXY adjusted USD closes
RATES_FACTORS_CSV=None # optional exact ERP Rates factor_returns.csv
RATES_CURVE_PATH=None # optional original Kim-Wright source
RATES_FUNDING_PATH=None # optional original FEDFUNDS CSV


## 2. Published Equity data, baseline v0.10 (unchanged)

The baseline remains untouched. Factor portfolios are fixed after formation, not continuously traded at zero cost. Neutrality against model factors holds in training, not necessarily OOS.

In [3]:
data=EquityData.from_public_sources(cache_dir=str(CACHE),aqr=True,aqr_paths=AQR_PATHS)
base=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
    validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,n_pcs=N_PCS)
print('Training/OOS months:',base.metadata['formation_months'],base.metadata['oos_months'])
print('Original residual PC premia:');display(base.fmp_premia_tests.round(5))
print('Baseline mean-pricing errors / attribution:');display(base.holdout_metrics.round(5))


Training/OOS months: 510 248
Original residual PC premia:


,train_spanning_alpha_ann,oos_spanning_alpha_ann,oos_spanning_t_hac,oos_spanning_p_hac,oos_spanning_r2,oos_mean_ann,oos_mean_t_hac,oos_mean_p_hac,oos_empirical_factor_beta_max,n_oos_months,eigenvalue_share_train,oos_spanning_p_holm,oos_mean_p_holm
factor,,,,,,,,,,,,,
RPC1,0.00727,0.00292,0.16200,0.87130,0.04010,-0.00086,-0.04638,0.96301,0.21522,248,0.14545,1.00000,1.00000
RPC2,-0.00983,0.00317,0.28001,0.77947,0.13041,0.01380,1.31182,0.18958,0.10023,248,0.08712,1.00000,0.94791
RPC3,0.00096,0.00899,0.87509,0.38153,0.07421,0.01320,1.30481,0.19196,0.10747,248,0.06813,1.00000,0.94791
RPC4,0.00227,0.00114,0.19062,0.84883,0.25542,0.00105,0.15787,0.87456,0.08634,248,0.06352,1.00000,1.00000
RPC5,0.00059,0.00621,1.45374,0.14602,0.27905,0.00244,0.43010,0.66712,0.10871,248,0.04826,0.73009,1.00000


Baseline mean-pricing errors / attribution:


,model,n_heldout_assets,n_test_months,oos_rmse_monthly,oos_r2_vs_ff_baseline,oos_realized_risk_attribution_r2,pricing_rmse_train_factor_means_ann,pricing_rmse_realized_factor_means_ann
n_new_pcs,,,,,,,,
0,FF_ONLY,50,248,0.01880,0.00000,0.89666,0.03349,0.01306
1,FF_PLUS_1_RPC,50,248,0.01882,-0.00210,0.89645,0.03312,0.01305
2,FF_PLUS_2_RPC,50,248,0.01882,-0.00216,0.89644,0.03353,0.01298
3,FF_PLUS_3_RPC,50,248,0.01880,-0.00032,0.89663,0.03355,0.01299
4,FF_PLUS_4_RPC,50,248,0.01877,0.00335,0.89701,0.03424,0.01260
5,FF_PLUS_5_RPC,50,248,0.01848,0.03384,0.90016,0.03432,0.01260


## 3. Test the existence of synchronous residual structure

Circular-shift each discovery-asset residual independently `NULL_DRAWS` times, preserving each asset's time path and autocorrelation while destroying synchronous cross-asset dependence. The rankwise eigenvalue null and conservative max-eigenvalue FWER comparison test common residual *covariance*, **not priced risks**. Compare principal subspaces between two formation-only halves to assess instability; this is not a formal confidence interval.

In [4]:
ext=analyze_equity_factor_extensions(data,base,n_pcs=N_PCS,null_draws=NULL_DRAWS,
    null_seed=NULL_SEED,rp_gamma=GAMMAS,gx_k=N_PCS)
print('Residual eigenvalue null:');display(ext.eigen_null.round(6))
print('Formation subspace stability:');display(ext.subspace_stability.round(5))


Residual eigenvalue null:


,eigenvalue,null_rank_95,null_rank_99,null_max_95_FWER,p_rank_exploratory,p_max_FWER_conservative,passes_max_95_FWER
factor,,,,,,,
RPC1,0.007295,0.006318,0.006333,0.006318,0.005,0.005,True
RPC2,0.004370,0.003556,0.003622,0.006318,0.005,1.000,False
RPC3,0.003417,0.002840,0.002860,0.006318,0.005,1.000,False
RPC4,0.003186,0.001858,0.001906,0.006318,0.005,1.000,False
RPC5,0.002421,0.001726,0.001743,0.006318,0.005,1.000,False


Formation subspace stability:


,n_pcs,formation_half_subspace_overlap_mean_cos2,null_interpretation
0,1,0.61099,Half formation subsamples only; no guarantee o...
1,2,0.44066,Half formation subsamples only; no guarantee o...
2,3,0.57403,Half formation subsamples only; no guarantee o...
3,4,0.54842,Half formation subsamples only; no guarantee o...
4,5,0.50839,Half formation subsamples only; no guarantee o...


## 4. Pricing-oriented residual factor challengers + SDF + three-pass

Formation covariance `C` is augmented by `gamma × a a′` using **training OLS alpha vector** `a` rather than zero-mean residual scores. This is an RP-PCA-*inspired* test: not the precise Lettau–Pelger procedure. Each direction is projected into net-dollar and FF/AQR-beta neutral portfolios using training betas. Mean and spanning-alpha tests use OOS months and HAC; Holm correction covers the whole predeclared gamma × PC family. Held-out asset means and SDF/GMM are also scored OOS.

Giglio–Xiu-style three passes: return PCA, cross-sectional lambda from training heldout assets, projection of **observed factors** onto latent factors. It is a point-estimate research diagnostic, not weak-factor robust inference.

In [5]:
print('Predeclared pricing-oriented portfolios:');display(ext.pricing_oriented_premia.round(5))
print('Heldout mean-pricing errors, regularized SDF:');display(ext.pricing_oriented_sdf.round(5))
print('Three-pass observed FF/AQR premia diagnostics:');display(ext.three_pass_observed.round(5))
print('Three-pass validation errors:');display(ext.three_pass_diagnostics.round(5))


Predeclared pricing-oriented portfolios:


,gamma,pc,eigenvalue_pricing_weighted_train,factor_train_alpha_ann,oos_mean_ann,oos_mean_t_hac,oos_mean_p_hac,oos_alpha_ann,oos_alpha_t_hac,oos_alpha_p_hac,oos_beta_max_abs,train_beta_max_abs,n_oos,holm_adjusted_oos_alpha_p_all_tests
factor,,,,,,,,,,,,,,
RP0_PC1,0.0,1,0.00730,0.00727,-0.00086,-0.04638,0.96301,0.00292,0.16200,0.87130,0.21522,0.0,248,1.0
RP0_PC2,0.0,2,0.00437,-0.00983,0.01380,1.31182,0.18958,0.00317,0.28001,0.77947,0.10023,0.0,248,1.0
RP0_PC3,0.0,3,0.00342,0.00096,0.01320,1.30481,0.19196,0.00899,0.87509,0.38153,0.10747,0.0,248,1.0
RP0_PC4,0.0,4,0.00319,0.00227,0.00105,0.15787,0.87456,0.00114,0.19062,0.84883,0.08634,0.0,248,1.0
RP0_PC5,0.0,5,0.00242,0.00059,0.00244,0.43010,0.66712,0.00621,1.45374,0.14602,0.10871,0.0,248,1.0
RP12_PC1,12.0,1,0.00754,0.01568,-0.00001,-0.00074,0.99941,0.00497,0.26505,0.79097,0.20161,0.0,248,1.0
RP12_PC2,12.0,2,0.00555,-0.01991,-0.00009,-0.01759,0.98597,-0.00422,-0.72869,0.46619,0.09246,0.0,248,1.0
RP12_PC3,12.0,3,0.00414,-0.00055,0.01624,1.64783,0.09939,0.00675,0.65047,0.51539,0.09528,0.0,248,1.0
RP12_PC4,12.0,4,0.00330,-0.00069,0.00586,0.66324,0.50718,0.00420,0.48874,0.62502,0.11534,0.0,248,1.0


Heldout mean-pricing errors, regularized SDF:


,gamma,n_priced_components,pricing_rmse_train_ann,pricing_rmse_oos_ann,pricing_mae_oos_ann,n_assets
0,0.0,0,0.01010,0.03041,0.02326,50
1,0.0,1,0.01011,0.03041,0.02323,50
2,0.0,5,0.00863,0.03045,0.02338,50
3,12.0,0,0.01010,0.03041,0.02326,50
4,12.0,1,0.01010,0.03041,0.02326,50
5,12.0,5,0.00747,0.03026,0.02349,50
6,60.0,0,0.01010,0.03041,0.02326,50
7,60.0,1,0.01010,0.03041,0.02326,50
8,60.0,5,0.00763,0.03022,0.02339,50


Three-pass observed FF/AQR premia diagnostics:


,gx_threepass_lambda_ann,factor_actual_train_mean_ann,factor_actual_oos_mean_ann
factor,,,
MKT,-0.01434,0.05709,0.10333
SMB,0.03133,0.03475,-0.00309
HML,0.04276,0.05613,-0.00762
RMW,0.02029,0.03015,0.03020
CMA,0.03042,0.04150,0.00396
MOM,0.10361,0.10167,0.00967
BAB,0.08275,0.11386,0.03898
QMJ,0.00819,0.04479,0.03078


Three-pass validation errors:


,n_latent,n_discovery,n_validation,n_formation,formation_cs_rmse_ann,oos_cs_rmse_ann,lambda_dispersion_monthly
0,5,80,50,510,0.01122,0.03006,0.15778


## 5. Download macro & cross-asset proxies automatically from FRED

`CPIAUCSL`: Consumer Price Index; `INDPRO`: Industrial Production; `DGS10` and `DGS2`: Treasury yields; `BAA10Y`: Moody's Baa–10Y Treasury spread; `VIXCLS`: CBOE volatility index. Monthly official observations are resampled without forward fill. Features: log YoY CPI/IP, 10Y–2Y slope, credit spread and log VIX. **First differences are statistical economic changes, not event-study identified macro surprises.**

The loader attempts a combined FRED download, then per-series downloads, then caches. If Colab cannot reach FRED, download the **original** combined CSV from `https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL%2CINDPRO%2CDGS10%2CDGS2%2CBAA10Y%2CVIXCLS`, place its path in `FRED_LOCAL_PATH`, and re-run this cell. Do not fill unavailable values or substitute synthetic data.
**Colab file uploads:** if you have uploaded `fredgraph (5).zip` (or any `fredgraph*.zip` / `fredgraph*.csv`) in the `/content` Files panel, the next cell automatically discovers it and bypasses FRED downloads. You may still set `FRED_LOCAL_PATH` explicitly. Upload the file before executing this cell.


In [6]:
# Select a complete Equity file, never accidentally a partial Rates download.
from pathlib import Path
macro=None
try:
    fred_path,rejected=find_complete_equity_fred_file([Path('/content'),Path.cwd(),CACHE/'macro'],explicit=FRED_LOCAL_PATH)
    print('Rejected local candidates:',rejected)
    if fred_path is not None: print('Using local FRED:',fred_path)
    try:
        macro=load_equity_fred_macro(cache_dir=CACHE/'macro',path=fred_path,as_of=AS_OF,timeout=8)
    except (ConnectionError,ValueError,FileNotFoundError) as exc:
        if not PROMPT_FRED_UPLOAD: raise
        try: from google.colab import files
        except ImportError: raise exc
        print('Upload the original combined six-series FRED CSV/ZIP (e.g. fredgraph (5).zip). Cancel to continue without FRED.')
        print('https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL,INDPRO,DGS10,DGS2,BAA10Y,VIXCLS')
        uploaded=files.upload()
        for filename in uploaded:
            try:
                path,_=find_complete_equity_fred_file([],explicit=filename)
                macro=load_equity_fred_macro(cache_dir=CACHE/'macro',path=path,as_of=AS_OF)
                break
            except (ValueError,OSError) as e: print('Rejected',filename,e)
    if macro is not None:
        print('FRED source:',macro.provenance['source']); display(macro.states.tail(6));display(macro.changes.tail(6))
    else: print('FRED not estimated; no data substituted.')
except (ConnectionError,FileNotFoundError,ValueError) as exc:
    print('FRED not estimated:',exc)


Rejected local candidates: []
Using local FRED: /content/fredgraph (5).zip
FRED source: user supplied FRED CSV/ZIP


,CPI_INFLATION_YOY_PCT,INDPRO_GROWTH_YOY_PCT,TERM_SPREAD_10Y2Y_PP,BAA_TREASURY_SPREAD_PP,LOG_VIX
2026-05-31,4.082150,1.696021,0.47,1.57,2.729159
2026-06-30,3.404900,1.456451,0.30,1.53,2.800325
2026-07-31,3.250452,1.242082,0.47,1.63,2.771964
2026-08-31,3.298029,1.436816,0.41,1.59,2.702703
2026-09-30,3.649827,2.119450,0.41,1.47,2.793616
2026-10-31,NaN,2.126905,0.38,1.53,2.966303


,CHANGE_INFLATION_YOY_PP,CHANGE_IP_GROWTH_YOY_PP,CHANGE_TREASURY_10Y_PP,CHANGE_BAA_SPREAD_PP,CHANGE_LOG_VIX
2026-05-31,0.372567,0.273448,0.05,-0.13,-0.097563
2026-06-30,-0.677249,-0.239570,-0.01,-0.04,0.071166
2026-07-31,-0.154449,-0.214368,0.31,0.10,-0.028362
2026-08-31,0.047577,0.194733,0.00,-0.04,-0.069261
2026-09-30,0.351798,0.682635,0.54,-0.12,0.090913
2026-10-31,NaN,0.007454,-0.35,0.06,0.172687


## 6. FF and RPC economic factor mapping + conditional premia

**Contemporaneous interpretation:** regress FF/AQR and RPC returns on *monthly changes* of macro/cross-asset variables using formation dates, then evaluate the estimated mapping on OOS months. A high R² does **not** identify a causal macro risk premium.

**Predictability:** state variables are lagged by two complete calendar months and a Ridge forecast of factor expected returns is refitted expanding-window. Its OOS R² is compared with an expanding unconditional mean. Data are latest revised FRED historical observations, **not point-in-time vintages**. The optional one-step SDF numbers using economic changes are **proxy-priced moment diagnostics**, not validated prices of economically identified macro risks.

In [7]:
macro_result=None
if macro is not None:
    macro_result=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
        validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,
        n_pcs=N_PCS,macro_shocks=macro.changes,state_predictors=macro.states,
        state_lag_months=MACRO_STATE_LAG_MONTHS)
    print('FF + RPC factor exposures to economic changes (NOT risk prices):')
    display(macro_result.macro_exposures.round(5))
    print('Cross-time stability / interpretation diagnostics:')
    display(macro_result.macro_diagnostics.round(5))
    print('Lagged macro-state factor return forecasting, OOS vs unconditional:')
    display(macro_result.state_scores.round(5))
    print('Tentative one-step macro proxy SDF — NOT identified shock risk prices:')
    display(macro_result.macro_sdf_pricing.round(5))
else:
    print('Macro mapping and forecast remain UNESTIMATED; no proxy or synthetic input used.')


FF + RPC factor exposures to economic changes (NOT risk prices):


,CHANGE_INFLATION_YOY_PP,CHANGE_IP_GROWTH_YOY_PP,CHANGE_TREASURY_10Y_PP,CHANGE_BAA_SPREAD_PP,CHANGE_LOG_VIX
equity_factor_or_rpc,,,,,
MKT,0.00647,-0.00329,-0.03374,-0.07387,-0.15398
SMB,-0.00603,-0.00072,-0.00761,-0.07982,-0.01065
HML,-0.01180,-0.00063,-0.00681,0.01048,0.04415
RMW,-0.02018,-0.00098,0.01309,0.07761,0.05605
CMA,-0.00525,0.00069,0.00424,0.00861,0.04465
MOM,0.01012,0.00072,-0.00858,0.09391,0.02463
BAB,-0.02868,0.00157,-0.00727,0.03757,0.08275
QMJ,-0.01051,0.00051,0.02347,0.08717,0.06223
RPC1,0.00597,-0.00332,0.00288,0.01488,0.00665


Cross-time stability / interpretation diagnostics:


,oos_macro_r2_vs_train_mean,oos_macro_residual_mean_ann,n_macro_train_months,n_macro_oos_months
MKT,0.55317,0.03994,191.0,246.0
SMB,-0.12452,-0.02918,191.0,246.0
HML,-0.27099,-0.06054,191.0,246.0
RMW,-1.32677,-0.01845,191.0,246.0
CMA,-0.20847,-0.03663,191.0,246.0
MOM,0.04659,-0.09942,191.0,246.0
BAB,-1.14226,-0.09348,191.0,246.0
QMJ,-0.07413,-0.03343,191.0,246.0
RPC1,-0.14243,-0.02050,191.0,246.0
RPC2,-0.12221,0.03301,191.0,246.0


Lagged macro-state factor return forecasting, OOS vs unconditional:


,n_oos,oos_r2_vs_expanding_mean,loss_gain_monthly_sq,loss_gain_hac_t,loss_gain_hac_p,state_lag_months,state_ridge
factor,,,,,,,
BAB,247,-0.09043,-0.00007,-1.33441,0.18207,2,10.0
CMA,247,-0.05539,-0.00002,-1.46993,0.14158,2,10.0
HML,247,-0.03598,-0.00004,-1.57391,0.11551,2,10.0
MKT,247,-0.00328,-0.00001,-0.07831,0.93758,2,10.0
MOM,247,-0.00517,-0.00001,-0.14638,0.88362,2,10.0
QMJ,247,-0.02790,-0.00002,-1.07368,0.28297,2,10.0
RMW,247,-0.03002,-0.00001,-0.95371,0.34023,2,10.0
RPC1,247,-0.02792,-0.00001,-1.22289,0.22137,2,10.0
RPC2,247,-0.00801,-0.00000,-0.22676,0.82061,2,10.0


Tentative one-step macro proxy SDF — NOT identified shock risk prices:


,macro_implied_premium_ann,realized_factor_mean_ann_oos,macro_unexplained_mean_ann_oos,validation_pricing_rmse_train_ann,validation_pricing_rmse_oos_ann,macro_cs_condition_scaled
equity_factor_or_rpc,,,,,,
MKT,0.09171,0.10328,0.01157,0.02546,0.02562,30.64404
SMB,0.01925,-0.00318,-0.02243,0.02546,0.02562,30.64404
HML,0.05360,-0.00797,-0.06156,0.02546,0.02562,30.64404
RMW,0.03092,0.03231,0.00139,0.02546,0.02562,30.64404
CMA,-0.00147,0.00567,0.00714,0.02546,0.02562,30.64404
MOM,0.02445,0.01051,-0.01393,0.02546,0.02562,30.64404
BAB,0.12251,0.03928,-0.08323,0.02546,0.02562,30.64404
QMJ,-0.02906,0.03208,0.06114,0.02546,0.02562,30.64404
RPC1,-0.04232,-0.00109,0.04123,0.02546,0.02562,30.64404


## 7. v0.11 baseline exports and interpretation

All CSVs are separate. Conclude only when residual eigenvalue significance, OOS pricing, and OOS conditional prediction each survive their own tests. Neither residual R² nor FF6 alpha alone proves a new priced factor. No TAA factor rotation alpha is claimed.

In [8]:
base.export(OUT/'equity_baseline_v010')
ext.export(OUT/'equity_extensions_v011')
if macro is not None:
    macro.export(OUT/'public_FRED_inputs')
    (OUT/'public_FRED_inputs'/'source_metadata.json').write_text(__import__('json').dumps(macro.provenance,indent=2))
    if macro_result is not None:macro_result.export(OUT/'equity_economic_mapping')
print('Saved to:',OUT.resolve())
for x in sorted(OUT.rglob('*.csv')):print(' -',x)


Saved to: /content/erp_equity_v013_results
 - erp_equity_v013_results/equity_baseline_v010/factor_neutral_weights.csv
 - erp_equity_v013_results/equity_baseline_v010/factor_neutrality_tests.csv
 - erp_equity_v013_results/equity_baseline_v010/holdout_oos_attribution.csv
 - erp_equity_v013_results/equity_baseline_v010/holdout_oos_pricing.csv
 - erp_equity_v013_results/equity_baseline_v010/incremental_premia_tests.csv
 - erp_equity_v013_results/equity_baseline_v010/lagged_state_forecasts.csv
 - erp_equity_v013_results/equity_baseline_v010/lagged_state_scores.csv
 - erp_equity_v013_results/equity_baseline_v010/macro_diagnostics.csv
 - erp_equity_v013_results/equity_baseline_v010/macro_exposures.csv
 - erp_equity_v013_results/equity_baseline_v010/macro_sdf_coefficients.csv
 - erp_equity_v013_results/equity_baseline_v010/macro_sdf_pricing.csv
 - erp_equity_v013_results/equity_baseline_v010/realized_fmp_returns.csv
 - erp_equity_v013_results/equity_baseline_v010/residual_eigenvalues.csv
 - er

## 8. Rolling vs static: are the residual factors stable, and is pricing orientation useful?

**Freeze only at each formation date:** rolling windows of 120 months, rebalance annually. Each FMP is dollar-neutral and neutral to formation FF/AQR betas, using actual discovery-portfolio excess returns. Signs are aligned to the *previously known* factor direction. Annualized OOS premia are **not** automatically priced systematic risks. Prespecified gamma=0 vs gamma=12, first 3 PCs; no tuning on OOS results.

In [9]:
ROLLING_WINDOW_MONTHS=120
ROLLING_REBALANCE_MONTHS=12
V012_N_PCS=3
V012_GAMMAS=(0.,12.)
pricing=analyze_equity_v012(data,base,formation_window=ROLLING_WINDOW_MONTHS,
    rebalance_months=ROLLING_REBALANCE_MONTHS,n_pcs=V012_N_PCS,
    gammas=V012_GAMMAS,bootstrap_draws=399)
print('Rolling PC premia:');display(pricing.rolling_premia.round(5))
print('First/last formation, turnover and FF neutrality:')
display(pricing.rolling_diagnostics.head(6).round(5))
display(pricing.rolling_diagnostics.tail(6).round(5))
pricing.export(OUT/'equity_v012_rolling_and_pricing')


Rolling PC premia:


,oos_mean_ann,oos_mean_t_hac,oos_mean_p_hac,oos_alpha_ann,oos_alpha_t_hac,oos_alpha_p_hac,oos_months
factor,,,,,,,
RP0_PC1,0.00534,0.43490,0.66363,0.00975,0.73747,0.46084,248
RP0_PC2,-0.00352,-0.48413,0.62829,-0.00515,-0.70653,0.47986,248
RP0_PC3,0.00586,0.72051,0.47121,0.00783,0.88767,0.37472,248
RP12_PC1,0.01126,1.14723,0.25129,0.01471,1.31673,0.18793,248
RP12_PC2,0.01749,2.44120,0.01464,0.01668,2.43603,0.01485,248
RP12_PC3,0.00213,0.27751,0.78139,-0.00365,-0.45656,0.64798,248


First/last formation, turnover and FF neutrality:


,formed_as_of,holding_from,holding_through,factor,gamma,eigenvalue,training_beta_max_abs,net_dollar,gross,l1_turnover_on_rebalance,rolling_subspace_overlap_mean_cos2,direction_cosine_with_previous_formation
0,2005-12-31,2006-01-31,2006-12-31,RP0_PC1,0.0,0.01492,0.0,-0.0,1.0,NaN,NaN,NaN
1,2005-12-31,2006-01-31,2006-12-31,RP0_PC2,0.0,0.00826,0.0,0.0,1.0,NaN,NaN,NaN
2,2005-12-31,2006-01-31,2006-12-31,RP0_PC3,0.0,0.00704,0.0,0.0,1.0,NaN,NaN,NaN
3,2005-12-31,2006-01-31,2006-12-31,RP12_PC1,12.0,0.02989,0.0,-0.0,1.0,NaN,NaN,NaN
4,2005-12-31,2006-01-31,2006-12-31,RP12_PC2,12.0,0.01208,0.0,-0.0,1.0,NaN,NaN,NaN
5,2005-12-31,2006-01-31,2006-12-31,RP12_PC3,12.0,0.00826,0.0,0.0,1.0,NaN,NaN,NaN


,formed_as_of,holding_from,holding_through,factor,gamma,eigenvalue,training_beta_max_abs,net_dollar,gross,l1_turnover_on_rebalance,rolling_subspace_overlap_mean_cos2,direction_cosine_with_previous_formation
120,2025-12-31,2026-01-31,2026-08-31,RP0_PC1,0.0,0.01265,0.0,-0.0,1.0,0.52416,0.89250,0.94203
121,2025-12-31,2026-01-31,2026-08-31,RP0_PC2,0.0,0.00901,0.0,0.0,1.0,0.49245,0.89250,0.92902
122,2025-12-31,2026-01-31,2026-08-31,RP0_PC3,0.0,0.00518,0.0,0.0,1.0,0.36749,0.89250,0.93180
123,2025-12-31,2026-01-31,2026-08-31,RP12_PC1,12.0,0.02102,0.0,0.0,1.0,0.46887,0.77204,0.93180
124,2025-12-31,2026-01-31,2026-08-31,RP12_PC2,12.0,0.01031,0.0,-0.0,1.0,0.51166,0.77204,0.89746
125,2025-12-31,2026-01-31,2026-08-31,RP12_PC3,12.0,0.00568,0.0,0.0,1.0,0.72746,0.77204,0.74102


PosixPath('erp_equity_v013_results/equity_v012_rolling_and_pricing')

## 9. FMB vs two-step GMM on precisely the same holdout assets and dates

Compare FF6/BAB/QMJ alone, FF+3 *plain residual PCs* (gamma=0), and FF+3 *augmented-mean PCs* (gamma=12). FMB formation betas and mean lambda estimates are fixed before the OOS period; two-step GMM uses shrinkage HAC weighting and the SDF pricing moments. Rolling FMP returns are **not** retrospectively inserted into the fixed-formation horse race. The bootstrap resamples **monthly loss differences** in common 12-month blocks; do not select gamma based on OOS results.

In [10]:
print('Common-sample pricing: annualized mean RMSE, FMB vs two-step GMM')
display(pricing.pricing_comparison.round(6))
print('FMB training lambda estimates and HAC t-stats:')
display(pricing.fmb_lambdas.round(5))
print('Paired block-bootstrap differences vs original FF model:')
display(pricing.bootstrap_comparison.round(7))


Common-sample pricing: annualized mean RMSE, FMB vs two-step GMM


,model,estimator,method,n_assets,n_factors,rank,condition_scaled,oos_mean_pricing_rmse_ann,train_end,oos_start,oos_end,oos_months,weight_shrinkage,ridge,formation_moment_rmse_ann,weight_condition,gmm_j_inference
0,FF_AQR,FMB,"Fama-MacBeth two-pass, fixed formation betas",50,8,9.0,524.934505,0.030197,2005-12-31,2006-01-31,2026-08-31,248,NaN,NaN,NaN,NaN,NaN
1,FF_AQR,GMM,two-step HAC-weighted shrinkage GMM SDF,50,8,NaN,NaN,0.030987,2005-12-31,2006-01-31,2026-08-31,248,0.5,0.0001,0.010160,46.631343,NOT_VALIDATED: shrinkage + overlapping charact...
2,FF_AQR_RP0,FMB,"Fama-MacBeth two-pass, fixed formation betas",50,11,12.0,623.517126,0.029832,2005-12-31,2006-01-31,2026-08-31,248,NaN,NaN,NaN,NaN,NaN
3,FF_AQR_RP0,GMM,two-step HAC-weighted shrinkage GMM SDF,50,11,NaN,NaN,0.031899,2005-12-31,2006-01-31,2026-08-31,248,0.5,0.0001,0.009553,46.855937,NOT_VALIDATED: shrinkage + overlapping charact...
4,FF_AQR_RP12,FMB,"Fama-MacBeth two-pass, fixed formation betas",50,11,12.0,684.135815,0.029815,2005-12-31,2006-01-31,2026-08-31,248,NaN,NaN,NaN,NaN,NaN
5,FF_AQR_RP12,GMM,two-step HAC-weighted shrinkage GMM SDF,50,11,NaN,NaN,0.030276,2005-12-31,2006-01-31,2026-08-31,248,0.5,0.0001,0.009364,47.047573,NOT_VALIDATED: shrinkage + overlapping charact...


FMB training lambda estimates and HAC t-stats:


,factor,lambda_training_ann,t_hac,p_hac,n_train_months,beta_partial_dispersion,model
0,intercept,0.03863,0.80552,0.42052,510,NaN,FF_AQR
1,MKT,0.02361,0.47757,0.63296,510,0.06016,FF_AQR
2,SMB,0.03046,1.66207,0.09650,510,0.46343,FF_AQR
3,HML,0.06265,2.28026,0.02259,510,0.11270,FF_AQR
4,RMW,0.02282,1.50557,0.13218,510,0.30421,FF_AQR
5,CMA,0.03931,2.93680,0.00332,510,0.25540,FF_AQR
6,MOM,0.17338,3.21949,0.00128,510,0.03789,FF_AQR
7,BAB,0.07446,1.62655,0.10383,510,0.06184,FF_AQR
8,QMJ,0.00271,0.14298,0.88631,510,0.12642,FF_AQR
9,intercept,0.10368,2.79764,0.00515,510,NaN,FF_AQR_RP0


Paired block-bootstrap differences vs original FF model:


,model,estimator,mse_improvement_monthly,block_bootstrap_ci_low,block_bootstrap_ci_high,ci_excludes_zero,draws,block_months
0,FF_AQR_RP0,FMB,2.000000e-07,-1.000000e-07,4.000000e-07,False,399,12
1,FF_AQR_RP12,FMB,2.000000e-07,-0.000000e+00,3.000000e-07,False,399,12
2,FF_AQR_RP0,GMM,-4.000000e-07,-2.500000e-06,1.700000e-06,False,399,12
3,FF_AQR_RP12,GMM,3.000000e-07,-3.000000e-07,9.000000e-07,False,399,12


## 10. Actual FOMC policy-information shocks — Jarociński–Karadi

The updated [author dataset](https://github.com/marekjarocinski/jkshocks_update_fed) contains monthly *MP_median* and *CBI_median* high-frequency announcement shocks. The two add to the announcement policy surprise. The panel is monthly since 1990, **not the same as changes in the level of the policy rate**; source currently ends March 2026, so months after that are missing, not zero. First attempt public GitHub, then cached copy, then a local original CSV (`shocks_fed_jk_m.csv`). This is independent of FRED macro download.

**Same-month fit** measures shock-factor comovement and is not a tradeable month-ahead predictor. The lag-1 experiment tests whether shocks observed during month t predict factor returns in month t+1. These are observational factor-response diagnostics, not yet an identified macro risk-premium estimator.
**Hotfix:** Original-source JK months with failed decomposition are excluded as missing (not zero) and recorded in shock provenance.


In [11]:
from pathlib import Path
# Optional, if GitHub unavailable: upload author CSV and use its actual path here.
JK_LOCAL_PATH=None
if JK_LOCAL_PATH is None:
    candidates=sorted(Path('/content').glob('shocks_fed_jk_m*.csv')) if Path('/content').exists() else []
    if candidates:
        JK_LOCAL_PATH=str(candidates[-1]);print('Found uploaded JK file:',JK_LOCAL_PATH)
jk=None;shock_mapping=None
try:
    jk=load_jk_monetary_shocks(cache_dir=CACHE/'shock_data',path=JK_LOCAL_PATH,as_of=AS_OF)
    print('JK source:',jk.provenance['source'],'ends',jk.shocks.index.max())
    print('Source months excluded (unknown shocks, not zero):',jk.provenance.get('excluded_invalid_decomposition_months',[]))
    f,_=data.factors(MODEL)
    shock_mapping=analyze_equity_shock_mapping(f,pd.DataFrame(index=f.index),jk,
        train_end=TRAIN_END)
    if shock_mapping.empty:
        print('Insufficient common train/OOS months for JK analysis')
    else:
        display(shock_mapping.round(5))
    jk.export(OUT/'jk_announcement_shocks')
    shock_mapping.to_csv(OUT/'jk_legacy_ff_exposure_and_predictability.csv',index=False)
    # Pricing: nontraded-shock beta prices inferred cross-sectionally, FF controls.
    # This is a point-estimate diagnostic under high-frequency identifying assumptions.
    _,rf=data.factors(MODEL)
    holdout=pd.concat([data.excess_returns(k,rf).add_prefix(k+'__')
         for k in VALIDATION],axis=1)
    shock_price_summary,shock_price_lambda=estimate_jk_shock_prices(
         holdout,f,jk,train_end=TRAIN_END)
    print('JK nontraded-shock prices, heldout common sample (exploratory):')
    display(shock_price_summary.round(6))
    display(shock_price_lambda.loc[shock_price_lambda.factor.isin(
         ['MP_median','CBI_median'])].round(6))
    shock_price_summary.to_csv(OUT/'jk_common_sample_mean_pricing.csv',index=False)
    shock_price_lambda.to_csv(OUT/'jk_shock_fmb_lambda_hac.csv',index=False)
except (ConnectionError,ValueError,FileNotFoundError) as e:
    print('JK shocks unavailable: ',e)
    print('Download https://raw.githubusercontent.com/marekjarocinski/jkshocks_update_fed/main/shocks_fed_jk_m.csv')
    print('Upload the original CSV and re-run this cell; FRED levels are not used as a substitute.')


/content/erp_v013_notebook_runtime/src/erp/equity_shocks.py:85: RuntimeWarning: JK source contains 4 months without a valid median decomposition: 2008-01-31, 2008-10-31, 2010-05-31, 2020-03-31. These are set to NaN (NOT zero); they are excluded from regressions.
  panel=_jk_parse(content)


JK source: live author-maintained JK GitHub CSV ends 2026-09-30 00:00:00
Source months excluded (unknown shocks, not zero): ['2008-01-31', '2008-10-31', '2010-05-31', '2020-03-31']


,factor,shock_group,lag,train_r2,oos_r2_vs_train_mean,n_train,n_oos,oos_start,oos_end,lag_months,interpretation,exposure_MP_median,t_hac_MP_median,exposure_CBI_median,t_hac_CBI_median
0,MKT,JK_MP_CBI,0,0.08562,0.02970,191,244,2006-01-31,2026-08-31,0,contemporaneous association,-0.21126,-5.62998,-0.02781,-0.34608
1,MKT,JK_MP_CBI,1,0.02092,-0.02615,190,244,2006-01-31,2026-08-31,1,forecast using prior month event shocks,0.07262,1.34546,-0.19145,-1.19582
2,SMB,JK_MP_CBI,0,0.00012,0.00198,191,244,2006-01-31,2026-08-31,0,contemporaneous association,-0.00573,-0.14958,0.00865,0.09651
3,SMB,JK_MP_CBI,1,0.02991,-0.03076,190,244,2006-01-31,2026-08-31,1,forecast using prior month event shocks,-0.09997,-3.30533,-0.03325,-0.42805
4,HML,JK_MP_CBI,0,0.06005,-0.04507,191,244,2006-01-31,2026-08-31,0,contemporaneous association,0.13016,4.25165,-0.13744,-2.04070
5,HML,JK_MP_CBI,1,0.04399,-0.05867,190,244,2006-01-31,2026-08-31,1,forecast using prior month event shocks,-0.11277,-2.40182,0.11146,0.94324
6,RMW,JK_MP_CBI,0,0.02379,-0.05615,191,244,2006-01-31,2026-08-31,0,contemporaneous association,0.08083,1.93892,-0.10528,-1.25520
7,RMW,JK_MP_CBI,1,0.00887,-0.01724,190,244,2006-01-31,2026-08-31,1,forecast using prior month event shocks,-0.05425,-1.27721,0.02355,0.17831
8,CMA,JK_MP_CBI,0,0.06641,-0.02192,191,244,2006-01-31,2026-08-31,0,contemporaneous association,0.10275,3.68153,-0.11006,-1.85312
9,CMA,JK_MP_CBI,1,0.06708,-0.11174,190,244,2006-01-31,2026-08-31,1,forecast using prior month event shocks,-0.07521,-2.42576,0.19201,2.70170


JK nontraded-shock prices, heldout common sample (exploratory):


,model,method,n_assets,n_factors,rank,condition_scaled,oos_mean_pricing_rmse_ann,formation_months,oos_months,oos_start,oos_end
0,FF_AQR,"Fama-MacBeth two-pass, fixed formation betas",50,8,9,173.296074,0.037320,191,244,2006-01-31,2026-08-31
1,FF_AQR_PLUS_JK,"Fama-MacBeth two-pass, fixed formation betas",50,10,11,232.820061,0.037847,191,244,2006-01-31,2026-08-31


,factor,lambda_training_ann,t_hac,p_hac,n_train_months,beta_partial_dispersion,model
18,MP_median,-0.031165,-0.177013,0.859498,191,0.030384,FF_AQR_PLUS_JK
19,CBI_median,0.057601,1.101441,0.270705,191,0.083364,FF_AQR_PLUS_JK


## 11. Inflation *release* surprises — optional separate input

There is no reliable automatic public consensus archive in this release. **Do not substitute** FRED CPI changes or Cleveland Fed expectations for verified announcement surprises. If you possess release-level data with `release_date,actual,consensus,consensus_source` (same rate units, pre-release consensus), load the CSV/ZIP. Results are separate from monetary-policy shocks. The parser cannot independently verify when a provider recorded its consensus.

In [12]:
CPI_RELEASE_CSV_PATH=None
inflation_shocks=None
if CPI_RELEASE_CSV_PATH is not None:
    inflation_shocks=load_inflation_release_surprises(CPI_RELEASE_CSV_PATH,as_of=AS_OF)
    ff,_=data.factors(MODEL)
    cpi_exposures=analyze_equity_shock_mapping(ff,pricing.rolling_returns,
        inflation_shocks,train_end=TRAIN_END,min_oos=24)
    display(cpi_exposures.round(5))
    inflation_shocks.export(OUT/'inflation_release_surprises')
else:
    print('CPI announcement surprises NOT estimated: no verified actual/consensus release file')


CPI announcement surprises NOT estimated: no verified actual/consensus release file


## 12. Dedicated rolling-factor economic test — new in v0.12.2

All six rolling factors are now genuinely observed during the dedicated 2006–2011 mapping training period. Use a common complete-case calendar for FF, RP, holdout assets and current/lagged shocks. Pricing compares FF, FF+JK, FF+each RP family and FF+each RP family+JK under FMB and GMM. The shock-only SDF contribution is a descriptive covariance-pricing projection calibrated on the holdout portfolios, not structural causal decomposition. Historical OOS dates are reused from earlier research. HAC uses consecutive complete-case observations; weak-identification inference is not validated.

In [13]:
ff,rf=data.factors(MODEL)
holdout=pd.concat([data.excess_returns(k,rf).add_prefix(k+'__') for k in VALIDATION],axis=1)
rolling_economic={}
economic_inputs={}
if jk is not None: economic_inputs['JK']=jk.shocks[['MP_median','CBI_median']]
if macro is not None: economic_inputs['FRED_PROXY']=macro.changes
for label,shock_panel in economic_inputs.items():
    try:
        result=analyze_rolling_shocks(ff,pricing.rolling_returns,holdout,shock_panel,
            train_end=SHOCK_TRAIN_END,as_of=AS_OF,group_label=label)
        rolling_economic[label]=result
        result.export(OUT/('rolling_economic_'+label))
        print(label,'common sample:',result.metadata)
        print('Factor mapping (all FF and RP, including FF-controlled RP):');display(result.mapping.round(5))
        print('Heldout pricing on identical dates:');display(result.pricing.round(6))
        print('FMB risk prices (exploratory):');display(result.lambdas[result.lambdas.factor.isin(shock_panel.columns)].round(6))
        print('Shock-only SDF implied premia, not causal attribution:');display(result.premium_mapping.round(5))
    except ValueError as e: print(label,'dedicated RP economic study not estimated:',e)
if not rolling_economic: print('No dedicated economic test completed; inspect input/coverage messages.')


JK common sample: {'train_start': '2006-01-31', 'train_end': '2011-12-31', 'oos_start': '2012-01-31', 'oos_end': '2026-08-31', 'n_train': 66, 'n_oos': 174, 'requested_cutoff': '2011-12-31', 'excluded_months': 8, 'inference': 'HAC and paired 12-observation block bootstrap on complete-case observations, conditional on fixed formation estimates; no training uncertainty or multiple-test adjustment; missing calendar months compress HAC lags. No Shanken or weak identification correction; GMM shrinkage no standard J-test.', 'economic_group': 'JK', 'design': 'Retrospective common-sample diagnostic; OOS period already inspected in prior releases; no new independent confirmation.'}
Factor mapping (all FF and RP, including FF-controlled RP):


,factor,specification,n_train,n_oos,train_r2,oos_r2_vs_train_mean,beta_MP_median,t_MP_median,p_MP_median,beta_CBI_median,...,p_CBI_median,oos_r2_increment_vs_FF,train_alpha_FF_ann,train_alpha_FF_economic_ann,oos_expost_alpha_FF_ann,oos_expost_alpha_FF_t,oos_expost_alpha_FF_p,oos_expost_alpha_FF_economic_ann,oos_expost_alpha_FF_economic_t,oos_expost_alpha_FF_economic_p
0,MKT,contemporaneous,66,174,0.09888,0.04151,-0.23685,-4.13521,0.00004,0.15823,...,0.36022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MKT,lag1,66,174,0.08764,-0.07138,0.07349,0.67208,0.50153,0.48723,...,0.08852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SMB,contemporaneous,66,174,0.01989,0.01344,-0.02258,-0.41365,0.67913,0.10695,...,0.06892,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMB,lag1,66,174,0.05489,-0.07718,-0.04340,-0.86345,0.38789,0.18508,...,0.15509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HML,contemporaneous,66,174,0.03867,-0.01442,0.01416,0.44310,0.65769,0.19688,...,0.07349,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,HML,lag1,66,174,0.10579,-0.00080,0.16483,2.20695,0.02732,0.19317,...,0.31541,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RMW,contemporaneous,66,174,0.01297,-0.01126,0.03133,1.45256,0.14635,-0.01170,...,0.84836,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,RMW,lag1,66,174,0.00807,0.00451,-0.01419,-0.45550,0.64875,0.03909,...,0.46377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CMA,contemporaneous,66,174,0.10390,-0.02794,0.08490,5.69856,0.00000,0.03266,...,0.52260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,CMA,lag1,66,174,0.01607,-0.00718,0.02283,0.67928,0.49696,-0.04098,...,0.63377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Heldout pricing on identical dates:


,model,estimator,n_train,n_oos,method,n_assets,n_factors,rank,condition_scaled,oos_mean_pricing_rmse_ann,weight_shrinkage,ridge,formation_moment_rmse_ann,weight_condition,gmm_j_inference,comparison_parent,rmse_improvement_ann,paired_mean_rmse_ci_low,paired_mean_rmse_ci_high
0,FF,FMB,66,174,"Fama-MacBeth two-pass, fixed formation betas",50,8,9.0,295.812888,0.038927,NaN,NaN,NaN,NaN,NaN,FF,0.000000,0.000000,0.000000
1,FF,GMM,66,174,two-step HAC-weighted shrinkage GMM SDF,50,8,NaN,NaN,0.040243,0.5,0.0001,0.016904,44.819681,NOT_VALIDATED: shrinkage + overlapping charact...,FF,0.000000,0.000000,0.000000
2,FF_JK,FMB,66,174,"Fama-MacBeth two-pass, fixed formation betas",50,10,11.0,374.086908,0.039044,NaN,NaN,NaN,NaN,NaN,FF,-0.000117,-0.000289,0.000082
3,FF_JK,GMM,66,174,two-step HAC-weighted shrinkage GMM SDF,50,10,NaN,NaN,0.040531,0.5,0.0001,0.016711,45.212810,NOT_VALIDATED: shrinkage + overlapping charact...,FF,-0.000288,-0.000589,0.000141
4,FF_RP0,FMB,66,174,"Fama-MacBeth two-pass, fixed formation betas",50,11,12.0,294.222527,0.038892,NaN,NaN,NaN,NaN,NaN,FF,0.000034,-0.000257,0.000332
5,FF_RP0,GMM,66,174,two-step HAC-weighted shrinkage GMM SDF,50,11,NaN,NaN,0.040919,0.5,0.0001,0.017100,44.189275,NOT_VALIDATED: shrinkage + overlapping charact...,FF,-0.000676,-0.001384,0.001213
6,FF_RP0_JK,FMB,66,174,"Fama-MacBeth two-pass, fixed formation betas",50,13,14.0,371.256786,0.039008,NaN,NaN,NaN,NaN,NaN,FF_RP0,-0.000115,-0.000285,0.000078
7,FF_RP0_JK,GMM,66,174,two-step HAC-weighted shrinkage GMM SDF,50,13,NaN,NaN,0.040828,0.5,0.0001,0.016800,44.571972,NOT_VALIDATED: shrinkage + overlapping charact...,FF_RP0,0.000091,-0.000739,0.000621
8,FF_RP12,FMB,66,174,"Fama-MacBeth two-pass, fixed formation betas",50,11,12.0,298.616275,0.038915,NaN,NaN,NaN,NaN,NaN,FF,0.000012,-0.000106,0.000100
9,FF_RP12,GMM,66,174,two-step HAC-weighted shrinkage GMM SDF,50,11,NaN,NaN,0.039867,0.5,0.0001,0.016609,44.733634,NOT_VALIDATED: shrinkage + overlapping charact...,FF,0.000376,-0.000941,0.001028


FMB risk prices (exploratory):


,factor,lambda_training_ann,t_hac,p_hac,n_train_months,beta_partial_dispersion,model
18,MP_median,-0.016841,-0.158068,0.874403,66,0.039624,FF_JK
19,CBI_median,0.059456,0.856378,0.391789,66,0.054233,FF_JK
44,MP_median,0.005617,0.049577,0.960459,66,0.056237,FF_RP0_JK
45,CBI_median,0.049454,0.723962,0.469089,66,0.055113,FF_RP0_JK
70,MP_median,-0.031370,-0.263229,0.792374,66,0.043507,FF_RP12_JK
71,CBI_median,0.056581,0.756764,0.449191,66,0.058189,FF_RP12_JK


... (6 outputs left out)

FMB risk prices (exploratory):


,factor,lambda_training_ann,t_hac,p_hac,n_train_months,beta_partial_dispersion,model
18,CHANGE_INFLATION_YOY_PP,2.490911,1.764257,0.077689,72,0.002455,FF_FRED_PROXY
19,CHANGE_IP_GROWTH_YOY_PP,0.962073,0.231113,0.817227,72,0.001299,FF_FRED_PROXY
20,CHANGE_TREASURY_10Y_PP,0.472900,0.848038,0.396417,72,0.008474,FF_FRED_PROXY
21,CHANGE_BAA_SPREAD_PP,0.380233,0.462027,0.644062,72,0.008192,FF_FRED_PROXY
22,CHANGE_LOG_VIX,-0.498324,-1.527886,0.126541,72,0.012598,FF_FRED_PROXY
47,CHANGE_INFLATION_YOY_PP,2.098069,1.483991,0.137811,72,0.002502,FF_RP0_FRED_PROXY
48,CHANGE_IP_GROWTH_YOY_PP,0.281553,0.064012,0.948961,72,0.001325,FF_RP0_FRED_PROXY
49,CHANGE_TREASURY_10Y_PP,0.489050,0.934777,0.349903,72,0.008398,FF_RP0_FRED_PROXY
50,CHANGE_BAA_SPREAD_PP,0.399332,0.530094,0.596047,72,0.009317,FF_RP0_FRED_PROXY
51,CHANGE_LOG_VIX,-0.495254,-1.620848,0.105050,72,0.013395,FF_RP0_FRED_PROXY


Shock-only SDF implied premia, not causal attribution:


,factor,shock_only_sdf_implied_ann,formation_mean_ann,oos_realized_mean_ann,contribution_CHANGE_INFLATION_YOY_PP_ann,contribution_CHANGE_IP_GROWTH_YOY_PP_ann,contribution_CHANGE_TREASURY_10Y_PP_ann,contribution_CHANGE_BAA_SPREAD_PP_ann,contribution_CHANGE_LOG_VIX_ann
0,MKT,0.03747,0.02402,0.13711,0.04174,-0.00229,0.05817,-0.19279,0.13265
1,SMB,0.00728,0.02607,-0.01523,0.00759,-0.00100,0.03195,-0.05822,0.02695
2,HML,0.01753,-0.03000,-0.00051,0.01415,-0.00163,0.01099,-0.01286,0.00688
3,RMW,-0.00381,0.07095,0.01620,-0.00405,0.00084,-0.01467,0.03666,-0.02259
4,CMA,-0.00503,0.01948,-0.00033,-0.01099,-0.00079,-0.00043,0.00718,-0.00001
5,MOM,0.08985,-0.02558,0.02726,0.01847,0.00114,-0.01734,0.12229,-0.03471
6,BAB,-0.01994,-0.02295,0.06499,0.02622,-0.00079,0.02726,-0.07244,-0.00018
7,QMJ,0.00076,0.05319,0.02231,-0.02472,0.00244,-0.03867,0.11035,-0.04864
8,RP0_PC1,0.01992,0.02205,-0.00224,0.02060,-0.00017,0.01201,-0.01351,0.00099
9,RP0_PC2,0.00173,0.01429,-0.01001,0.00361,0.00018,-0.00121,-0.00303,0.00218


## 13. Constrained RP-PCA: objective and stability

Use the balanced-panel Lettau–Pelger moment matrix $R^{\top}R/T+\gamma\bar r\bar r^{\top}$ (gamma=-1 is centered PCA). Here the asset-weight space is restricted **before** extraction to zero-net-dollar and FF-neutral directions. This is a constrained adaptation, not a claim of complete published asymptotic inference. Gamma -1 and 11 match the legacy mean weights 0 and 12; normalization and assignment differ. Re-estimate on trailing 120 months annually; match columns by maximum absolute overlap to prior formation. Preserve all old RP returns. Report HAC alpha and Holm across the six new candidates (retrospective).

In [14]:
ff_all,discovery_all,holdout_all,_=_aligned(data,base)
crp,crp_diag,crp_w=constrained_rp_rolling(data,base,window=120,step=12,k=3,gammas=(-1.,11.))
crp.to_csv(OUT/'constrained_rp_returns.csv');crp_diag.to_csv(OUT/'constrained_rp_diagnostics.csv',index=False);crp_w.to_csv(OUT/'constrained_rp_weights.csv')
rows=[]
for col in crp:
    a=_hac_alpha(crp[col],ff_all);m=_hac_alpha(crp[col]);rows.append({'factor':col,'mean_ann':12*m['alpha_monthly'],'alpha_ann':12*a['alpha_monthly'],'t_hac':a['t_hac'],'p_hac':a['p_hac'],'nobs':a['nobs']})
crp_premia=pd.DataFrame(rows).set_index('factor');crp_premia['holm_p_new_candidates']=_holm(crp_premia.p_hac)
crp_premia.to_csv(OUT/'constrained_rp_premia.csv');display(crp_premia.round(5));display(crp_diag.tail(6))
rp_all=pd.concat([pricing.rolling_returns,crp],axis=1)


,mean_ann,alpha_ann,t_hac,p_hac,nobs,holm_p_new_candidates
factor,,,,,,
CRP-1_PC1,0.00990,0.01201,0.85895,0.39037,248,1.0
CRP-1_PC2,-0.00090,-0.00517,-0.74450,0.45657,248,1.0
CRP-1_PC3,0.00563,0.00678,0.79641,0.42580,248,1.0
CRP11_PC1,-0.00591,-0.01125,-1.00766,0.31362,248,1.0
CRP11_PC2,0.00315,0.00123,0.13479,0.89277,248,1.0
CRP11_PC3,-0.00846,-0.00481,-0.67907,0.49709,248,1.0


,formation,holding_from,factor,eigenvalue,subspace_stability,max_constraint_error,turnover
120,2024-12-31,2025-01-31,CRP11_PC1,0.011292,0.905907,8.326673e-17,0.589980
121,2024-12-31,2025-01-31,CRP11_PC2,0.007834,0.905907,1.526557e-16,0.376127
122,2024-12-31,2025-01-31,CRP11_PC3,0.003834,0.905907,2.949030e-17,0.376444
123,2025-12-31,2026-01-31,CRP11_PC1,0.012931,0.883384,6.245005e-17,0.450184
124,2025-12-31,2026-01-31,CRP11_PC2,0.006711,0.883384,8.326673e-17,0.485122
125,2025-12-31,2026-01-31,CRP11_PC3,0.004813,0.883384,9.367507e-17,0.445697


## 14. Three-pass prices (Giglio–Xiu-style) and uncertainty

Centered PCA of discovery returns; no-intercept cross-sectional pricing of latent betas on formation holdout returns; projection of observed factors on latent innovations. Three fixed latent dimensions, synchronized blocks with all three stages refit. Projection R² and condition number diagnose identification, not a formal weak-factor-robust test. Bootstrap intervals are exploratory, not published asymptotic inference.

In [15]:
gx_initial=gx_initial_inference(ff_all,rp_all,holdout_all,discovery_all,k_values=(5,8,12),train_end=SHOCK_TRAIN_END,as_of=AS_OF,draws=DYNAMIC_BOOT_DRAWS)
gx_initial.to_csv(OUT/'gx_initial_block_inference.csv',index=False);display(gx_initial.round(5))


,lambda_ann,ci_low_ann,ci_high_ann,projection_r2,condition,successful_draws,factor,k,n_train,train_end
0,0.01976,-0.16482,0.20361,0.99192,25.13763,199,MKT,5,72,2011-12-31
1,0.03300,-0.01756,0.08814,0.93272,25.13763,199,SMB,5,72,2011-12-31
2,-0.01795,-0.07273,0.06694,0.46002,25.13763,199,HML,5,72,2011-12-31
3,0.03681,-0.00157,0.07621,0.45957,25.13763,199,RMW,5,72,2011-12-31
4,0.00956,-0.01771,0.03866,0.20622,25.13763,199,CMA,5,72,2011-12-31
5,0.08514,-0.15613,0.32145,0.95505,25.13763,199,MOM,5,72,2011-12-31
6,-0.07292,-0.12060,0.02232,0.21930,25.13763,199,BAB,5,72,2011-12-31
7,0.04301,-0.09018,0.09225,0.80528,25.13763,199,QMJ,5,72,2011-12-31
8,-0.03943,-0.05240,0.08771,0.86249,25.13763,199,RP0_PC1,5,72,2011-12-31
9,-0.04049,-0.05340,0.02962,0.69920,25.13763,199,RP0_PC2,5,72,2011-12-31


## 15. Cross-asset traded-return inputs

Use original funded ERP Rates factors (Kim–Wright/FEDFUNDS) and a **separate** ETF proxy panel: IEF excess, HYG minus IEF (not duration-neutral), UUP excess (USD futures fund), DBC excess (commodity futures fund), VIXY excess (volatility futures fund, not a variance swap). All ETF inputs are adjusted USD close prices and all cash-funded legs subtract the French RF. No pre-inception backfill or substitute data. Source failures are logged; you can supply exact local CSVs. For revised macro states, add a separate managed-return experiment MKT/HML times CPI inflation and term slope with two-month lag and trailing standardization. No CPI release surprises are manufactured.

In [16]:
extra_panels={};extra_sources=[]
if jk is not None:extra_panels['JK']=(jk.shocks[['MP_median','CBI_median']],False)
if macro is not None:
    extra_panels['FRED_PROXY']=(macro.changes,False)
    extra_panels['MANAGED']=(managed_factor_returns(ff_all,macro.states),True)
if RUN_FUNDED_RATES:
    try:
        if RATES_FACTORS_CSV is not None:
            rate_factors=load_rates_factor_returns(RATES_FACTORS_CSV);rate_source=str(RATES_FACTORS_CSV)
        else:
            rd=RatesData.from_fred(cache_dir=CACHE/'rates',local_curve_path=RATES_CURVE_PATH,local_fedfunds_path=RATES_FUNDING_PATH)
            engine=RatesPremiaEngine(rd);engine.fit(as_of=AS_OF);rate_factors=engine.factor_returns.add_prefix('XA_RATES_');rate_source='ERP funded Kim-Wright / FEDFUNDS'
        extra_panels['FUNDED_RATES']=(rate_factors,True);extra_sources.append({'panel':'FUNDED_RATES','source':rate_source,'status':'available'})
    except (ConnectionError,OSError,ValueError,RuntimeError) as e:
        extra_sources.append({'panel':'FUNDED_RATES','status':'unavailable','detail':str(e)});print('Rates unavailable:',e,'; supply original ERP factor_returns.csv or source data.')
if RUN_CROSS_ASSET_ETFS:
    try:
        if ETF_ADJUSTED_CSV is None:
            candidate=Path('/content/etf_adjusted_prices.csv')
            if candidate.is_file(): ETF_ADJUSTED_CSV=str(candidate)
        if ETF_ADJUSTED_CSV is None:
            import importlib.util,subprocess,sys
            if importlib.util.find_spec('yfinance') is None:
                subprocess.run([sys.executable,'-m','pip','-q','install','yfinance'],check=True,timeout=120)
        px,pxsource=load_etf_adjusted_prices(cache_dir=CACHE/'cross_asset',path=ETF_ADJUSTED_CSV,as_of=AS_OF)
        xa=etf_factor_returns(px,rf,as_of=AS_OF);extra_panels['ETF_PROXY']=(xa,True);extra_sources.append({'panel':'ETF_PROXY','source':pxsource,'status':'available'})
    except Exception as e:
        extra_sources.append({'panel':'ETF_PROXY','status':'unavailable','detail':str(e)});print('ETF proxy data unavailable:',e,'; supply etf_adjusted_prices.csv with Date, IEF,HYG,UUP,DBC,VIXY adjusted closes.')
# A combined rates + ETFs experiment is distinct from the two standalone panels.
if 'FUNDED_RATES' in extra_panels and 'ETF_PROXY' in extra_panels:
    extra_panels['JOINT_XA']=(pd.concat([extra_panels['FUNDED_RATES'][0],extra_panels['ETF_PROXY'][0].drop(columns='XA_DURATION')],axis=1),True)
for name,(panel,traded) in extra_panels.items():
    panel.loc[:AS_OF].to_csv(OUT/(name+'_raw_factors.csv'));print(name,'complete months:',len(panel.loc[:AS_OF].dropna()),'traded-return proxy:',traded)
pd.DataFrame(extra_sources).to_csv(OUT/'cross_asset_sources.csv',index=False)


Treasury curve: loading one official Kim-Wright Board CSV (10 tenors).
Funding: verified FEDFUNDS mirror (868 months through 2026-10-31).
Kim-Wright curve ready: 443 months, 1990-01-31 to 2026-11-30
FEDFUNDS ready: 868 months, latest 2026-10-31
JK complete months: 436 traded-return proxy: False
FRED_PROXY complete months: 438 traded-return proxy: False
MANAGED complete months: 564 traded-return proxy: True
FUNDED_RATES complete months: 286 traded-return proxy: True
ETF_PROXY complete months: 187 traded-return proxy: True
JOINT_XA complete months: 187 traded-return proxy: True


## 16. Dynamic prices and posterior shrinkage — same forecast dates within each panel

Each quarterly observed origin uses historical data only. Fixed, expanding, and trailing 72-calendar-month estimates use the same forecast dates and holdout assets. Regressions estimate formation betas and FMB prices; GMM is two-step HAC-weighted shrinkage. FMB_SHRINK applies a zero-centered Gaussian prior with SD 0.25 times monthly factor volatility (intercept weakly regularized) to estimated prices using their HAC covariance. DIRECT_MEAN and DIRECT_SHRINK forecast traded FMP returns directly; no direct-mean target for nontraded shocks. GX_LATENT is a separate latent pricing model, not an FF regression. Extras are neutralized to FF on formation, preserving intercept; transformed return targets depend on hedge scheme. Mean-pricing RMSE, monthly forecast RMSE and factor forecasts are separate; bootstrap compares paired OOS mean errors conditional on fitted forecasts. Cross-panel scores are **not** directly comparable because start dates may differ. No trading strategy or costs yet.

In [17]:
dynamic_summaries=[]; dynamic_boots=[]; dynamic_factor_scores=[]; status=[]
# Full equity comparison including the two constrained challengers. Economic panels keep legacy RP0/RP12
# to isolate the effect of pricing dynamics and the new economic drivers.
experiments={'EQUITY':(None,False),**extra_panels}
for name,(panel,traded) in experiments.items():
    try:
        rp_input=rp_all if name=='EQUITY' else pricing.rolling_returns
        result=analyze_dynamic_premia(ff_all,rp_input,holdout_all,discovery_all,extras=panel,extra_traded=traded,
            as_of=AS_OF,window=DYNAMIC_WINDOW,min_train=DYNAMIC_MIN_TRAIN,step=DYNAMIC_REFIT,k=DYNAMIC_GX_K,draws=DYNAMIC_BOOT_DRAWS)
        folder=OUT/('dynamic_'+name);result.export(folder)
        dynamic_summaries.append(result.scores.assign(panel=name));dynamic_boots.append(result.bootstrap.assign(panel=name));dynamic_factor_scores.append(result.factor_scores.assign(panel=name))
        status.append({'panel':name,'status':'completed',**result.metadata});print(name,result.metadata['first_oos'],result.metadata['last_oos']);display(result.scores.round(5))
        if traded and panel is not None and name in ('ETF_PROXY','FUNDED_RATES','JOINT_XA'):
            try:
                econ=economic_return_decomposition(pd.concat([ff_all,pricing.rolling_returns],axis=1),panel,as_of=AS_OF,window=DYNAMIC_WINDOW,min_train=DYNAMIC_MIN_TRAIN,step=DYNAMIC_REFIT)
                econ.to_csv(folder/'raw_traded_economic_decomposition.csv',index=False)
                # Report modeled contributions separately from retained intercept. Not causal shares.
                agg=econ.groupby('factor')[['intercept_monthly','driver_implied_monthly','total_expected_monthly','actual_monthly']].mean()*12
                agg.to_csv(folder/'raw_traded_economic_summary_ann.csv');display(agg.round(5))
            except ValueError as e:print('Economic decomposition not estimated:',e)
        del result
    except (ValueError,np.linalg.LinAlgError) as e:
        status.append({'panel':name,'status':'not estimated','detail':str(e)});print(name,'not estimated:',e)
for name,frames in [('dynamic_all_scores',dynamic_summaries),('dynamic_all_bootstrap',dynamic_boots),('dynamic_all_factor_scores',dynamic_factor_scores)]:
    if frames:pd.concat(frames,ignore_index=True).to_csv(OUT/(name+'.csv'),index=False)
pd.DataFrame(status).to_csv(OUT/'dynamic_experiment_status.csv',index=False)
print('New experiment completion:');display(pd.DataFrame(status)[['panel','status']])


EQUITY 2012-01-31 00:00:00 2026-08-31 00:00:00


,scheme,model,estimator,n_months,mean_pricing_rmse_ann,monthly_forecast_rmse,mean_error_ann
0,expanding,FF,FMB,176,0.04502,0.05642,0.03916
1,expanding,FF,FMB_SHRINK,176,0.03632,0.05635,0.02997
2,expanding,FF,GMM,176,0.04378,0.05640,0.03760
3,expanding,FF_CRP-1,FMB,176,0.04504,0.05642,0.03916
4,expanding,FF_CRP-1,FMB_SHRINK,176,0.03489,0.05629,0.02864
5,expanding,FF_CRP-1,GMM,176,0.04417,0.05640,0.03813
6,expanding,FF_CRP11,FMB,176,0.04507,0.05642,0.03916
7,expanding,FF_CRP11,FMB_SHRINK,176,0.03447,0.05632,0.02853
8,expanding,FF_CRP11,GMM,176,0.04739,0.05643,0.04152
9,expanding,FF_RP0,FMB,176,0.04507,0.05642,0.03916


JK 2012-01-31 00:00:00 2026-08-31 00:00:00


,scheme,model,estimator,n_months,mean_pricing_rmse_ann,monthly_forecast_rmse,mean_error_ann
0,expanding,FF,FMB,175,0.02716,0.05384,0.01672
1,expanding,FF,FMB_SHRINK,175,0.02606,0.05383,0.01687
2,expanding,FF,GMM,175,0.02625,0.05382,0.01487
3,expanding,FF_EXTRA,FMB,175,0.02695,0.05384,0.01672
4,expanding,FF_EXTRA,FMB_SHRINK,175,0.02497,0.05382,0.01469
5,expanding,FF_EXTRA,GMM,175,0.02712,0.05383,0.01689
6,expanding,FF_RP0,FMB,175,0.02716,0.05384,0.01672
7,expanding,FF_RP0,FMB_SHRINK,175,0.02457,0.05378,0.01475
8,expanding,FF_RP0,GMM,175,0.02650,0.05383,0.01527
9,expanding,FF_RP12,FMB,175,0.02689,0.05384,0.01672


FRED_PROXY 2012-01-31 00:00:00 2026-08-31 00:00:00


,scheme,model,estimator,n_months,mean_pricing_rmse_ann,monthly_forecast_rmse,mean_error_ann
0,expanding,FF,FMB,174,0.04540,0.05664,0.03971
1,expanding,FF,FMB_SHRINK,174,0.03671,0.05658,0.03054
2,expanding,FF,GMM,174,0.04412,0.05663,0.03813
3,expanding,FF_EXTRA,FMB,174,0.04538,0.05664,0.03971
4,expanding,FF_EXTRA,FMB_SHRINK,174,0.03722,0.05658,0.03152
5,expanding,FF_EXTRA,GMM,174,0.04485,0.05663,0.03909
6,expanding,FF_RP0,FMB,174,0.04544,0.05664,0.03971
7,expanding,FF_RP0,FMB_SHRINK,174,0.03561,0.05650,0.02945
8,expanding,FF_RP0,GMM,174,0.04448,0.05664,0.03856
9,expanding,FF_RP12,FMB,174,0.04522,0.05664,0.03971


... (9 outputs left out)

,scheme,model,estimator,n_months,mean_pricing_rmse_ann,monthly_forecast_rmse,mean_error_ann
0,expanding,FF,FMB,115,0.02077,0.06271,-0.00204
1,expanding,FF,FMB_SHRINK,115,0.02544,0.06271,-0.01591
2,expanding,FF,GMM,115,0.02315,0.06271,-0.00372
3,expanding,FF_EXTRA,FMB,115,0.01985,0.06271,-0.00204
4,expanding,FF_EXTRA,FMB_SHRINK,115,0.02322,0.06272,-0.01304
5,expanding,FF_EXTRA,GMM,115,0.02059,0.06269,0.00006
6,expanding,FF_RP0,FMB,115,0.01980,0.06271,-0.00204
7,expanding,FF_RP0,FMB_SHRINK,115,0.02419,0.06273,-0.01456
8,expanding,FF_RP0,GMM,115,0.02137,0.06271,-0.00355
9,expanding,FF_RP12,FMB,115,0.01960,0.06271,-0.00204


,intercept_monthly,driver_implied_monthly,total_expected_monthly,actual_monthly
factor,,,,
BAB,0.07715,-0.01289,0.06426,0.02822
CMA,-0.00057,-0.00419,-0.00476,-0.00888
HML,0.00625,-0.02701,-0.02076,-0.01381
MKT,0.04335,0.08421,0.12756,0.13033
MOM,0.03087,-0.00990,0.02097,0.03063
QMJ,0.06612,-0.02148,0.04464,0.01553
RMW,0.03852,-0.00539,0.03312,0.02459
RP0_PC1,0.00127,-0.00206,-0.00079,0.00782
RP0_PC2,-0.01135,0.00227,-0.00908,-0.01034


New experiment completion:


,panel,status
0,EQUITY,completed
1,JK,completed
2,FRED_PROXY,completed
3,MANAGED,completed
4,FUNDED_RATES,completed
5,ETF_PROXY,completed
6,JOINT_XA,completed


## 17. Audit / next research decision

**Decision gates:** common residual risk, incremental OOS pricing after multiple-testing adjustment, pricing stability under rolling retraining, and pre-specified economic shock exposures/predictability must each be tested separately. Report uncertainty and any model selection as exploratory. This notebook does **not** claim published weak-factor robustness, identified inflation news when consensus is absent, or improved investable TAA value.

In [18]:
print('ERP',__version__)
print('Core output:',OUT.resolve())
print('Rolling premia factors:',len(pricing.rolling_premia))
print('Pricing candidates x methods:',len(pricing.pricing_comparison))
print('FMB lambda rows:',len(pricing.fmb_lambdas))
print('GMM shrinkage status: diagnostics only; no standard J-test')
print('Policy shock mapping:', 'available' if shock_mapping is not None else 'not estimated')
print('CPI release shocks:', 'available' if inflation_shocks is not None else 'not estimated')

print("Dedicated rolling economic studies completed:",list(rolling_economic))
print("FRED proxies available:",macro is not None)

print("Dynamic experiments:",[(x["panel"],x["status"]) for x in status])


ERP 0.13.0
Core output: /content/erp_equity_v013_results
Rolling premia factors: 6
Pricing candidates x methods: 6
FMB lambda rows: 33
GMM shrinkage status: diagnostics only; no standard J-test
Policy shock mapping: available
CPI release shocks: not estimated
Dedicated rolling economic studies completed: ['JK', 'FRED_PROXY']
FRED proxies available: True
Dynamic experiments: [('EQUITY', 'completed'), ('JK', 'completed'), ('FRED_PROXY', 'completed'), ('MANAGED', 'completed'), ('FUNDED_RATES', 'completed'), ('ETF_PROXY', 'completed'), ('JOINT_XA', 'completed')]
